In [1]:
%config IPCompleter.use_jedi = False
# %xmode Verbose
# %xmode context
%pdb off
%load_ext autoreload
%autoreload 3
# # Add exclusions for metaclass-using modules
# %aimport -neuropy.core.session.dataSession
# %aimport -neuropy.core.session.Formats.BaseDataSessionFormats
# %aimport -neuropy.core.session.Formats.Specific.KDibaOldDataSessionFormat
# %aimport -neuropy.core.session.Formats.Specific.BapunDataSessionFormat 
# %aimport -neuropy.core.session.Formats.Specific.RachelDataSessionFormat
# %aimport -neuropy.core.session.Formats.Specific.HiroDataSessionFormat

import sys
from pathlib import Path
import numpy as np
import pyvista as pv
import time
import sys
import random
import queue
import os
import pythreejs
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
import ipywidgets as widgets
from IPython.display import display, clear_output

# from .example_animated_3D_sensor_quality import EEGVisualizer
# from EXAMPLES.example_animated_3D_sensor_quality import EEGVisualizer

from phoemotivepocsensor3d.EEGVisualizer import EEGVisualizer

os.environ['QT_API'] = 'pyqt5'
os.environ['PYQTGRAPH_QT_LIB'] = 'PyQt5'

# from PyQt5.QtWebEngineWidgets import QWebEngineView ## this must come first, before any QtApplication is made: 'ImportError: QtWebEngineWidgets must be imported or Qt.AA_ShareOpenGLContexts must be set before a QCoreApplication instance is created'

# required to enable non-blocking interaction:
%gui qt5

# In your __init__ method
pv.set_jupyter_backend('trame')

Automatic pdb calling has been turned OFF


In [ ]:
# Create and display the visualizer
visualizer: EEGVisualizer = EEGVisualizer(model_path=r'C:\Users\pho\repos\EmotivEpoc\CyKit\EXTERNAL\meshes\CompleteEmotivEpocEEG.glb', update_interval=0.5, is_notebook=False)
# visualizer = EEGVisualizer(model_path=r'C:\Users\pho\repos\EmotivEpoc\CyKit\EXTERNAL\meshes\CompleteEmotivEpocEEG1.obj', update_interval=0.5)
visualizer.show()

Loading 3D model from: C:\Users\pho\repos\EmotivEpoc\CyKit\EXTERNAL\meshes\CompleteEmotivEpocEEG.glb
Created electrode sphere: AF3 at position [-0.02849366665483203, -0.0731334302412427, -0.06309279727001485]
Created electrode sphere: AF4 at position [0.02849366665483203, -0.07313343024976739, -0.06309279727001485]
Created electrode sphere: AuxCMS at position [-0.06634027689047482, -0.0030793339151980194, 0.012765326982607012]
Created electrode sphere: AuxDRL at position [0.06634027689047482, -0.0030793339151980194, 0.01276943341869375]
Created electrode sphere: CMS at position [-0.06563927965347782, -0.006030545767433053, -0.029458098261013078]
Created electrode sphere: DRL at position [0.06563927965347782, -0.006030545767433053, -0.029458098261013078]
Created electrode sphere: F3 at position [-0.02363707810785255, -0.05494754518077494, -0.0826450100923021]
Created electrode sphere: F4 at position [0.0236370781035902, -0.05494754518077494, -0.0826450100923021]
Created electrode sphere

In [ ]:
visualizer.update_visualization()

In [ ]:
visualizer.headset.array_names

In [ ]:
# Print the structure of the loaded model
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any




def extract_headset_mesh_electrode_data(headset_mesh: pv.MultiBlock, debug_print=False):
    electrode_poly_output_dict: Dict[str, pv.PolyData] = {}

    if debug_print:
        print(visualizer.headset)

    electrode_multiblock: pv.MultiBlock = headset_mesh.get_block(0)
    # If it's a MultiBlock, try listing the blocks
    if hasattr(electrode_multiblock, 'n_blocks'):
        # print(f"Number of blocks: {electrode_multiblock.n_blocks}")
        for i in range(electrode_multiblock.n_blocks):
            an_electrode_multiblock: pv.MultiBlock = electrode_multiblock.get_block(i)
            if debug_print:
                print(f"an_electrode_multiblock[{i}]: {type(an_electrode_multiblock)}")
            is_base_multiblock: bool = hasattr(an_electrode_multiblock, 'n_blocks') and (an_electrode_multiblock.n_blocks == 1)
            if is_base_multiblock:
                ## extract root
                # an_electrode_multiblock = an_electrode_multiblock.get_block(0) # PolyData
                an_electrode_polydata: pv.PolyData = an_electrode_multiblock.get_block(0).get_block(0) # PolyData
                electrode_poly_output_dict[i] = deepcopy(an_electrode_polydata)
                if debug_print:
                    print(f'\tFOUND BASE ELECTRODE POLYDATA:')
                    print(f'\t{an_electrode_polydata}')
            else:
                print(f'\tERR: FAILED TO FIND BASE ELECTRODE BLOCK:')
                print(f"\t\tNumber of blocks: {electrode_multiblock.n_blocks}")
                print(f'\t{an_electrode_multiblock}')
            

    ## OUTPUTS: electrode_poly_output_dict
    return electrode_poly_output_dict

electrode_poly_output_dict = extract_headset_mesh_electrode_data(headset_mesh=visualizer.headset)


In [ ]:
an_electrode_polydata.GetNamedFieldInformation()

In [ ]:
## INPUTS: electrode_poly_output_dict

self.electrode_names_dict = {
    0:"AF3",
    1:"AF4",
    2:"Arm_R_Body",
    3:"ArmL_Body",
    4:"AuxCMS",
    5:"AuxDRL",
    6:"CMS",
    7:"DRL",
    8:"F3",
    9:"F4",
    10:"F7",
    11:"F8",
    12:"FC5",
    13:"FC6",
    14:"Headset_Back_Body",
    15:"O1",
    16:"O2",
    17:"P7",
    18:"P8",
    19:"T7",
    20:"T8",	
}

assert len(self.electrode_names_dict) == len(electrode_poly_output_dict), f"len(electrode_names_dict): {len(self.electrode_names_dict)} != len(electrode_poly_output_dict): {len(electrode_poly_output_dict)}"

{self.electrode_names_dict[k]:v.center_of_mass().tolist() for k, v in electrode_poly_output_dict.items()}





# an_electrode_polydata.center_of_mass

In [ ]:
print(list(self.electrode_names_dict.values()))

# ['AF3', 'AF4', 'Arm_R_Body', 'ArmL_Body', 'AuxCMS', 'AuxDRL', 'CMS', 'DRL', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'Headset_Back_Body', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8']
['AF3', 'AF4', 'AuxCMS', 'AuxDRL', 'CMS', 'DRL', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8']

In [ ]:


an_electrode_multiblock = electrode_multiblock.get_block(0).get_block(0)

# electrode_multiblock
an_electrode_multiblock

In [ ]:

# If it's a MultiBlock, try listing the blocks
if hasattr(visualizer.headset, 'n_blocks'):
    print(f"Number of blocks: {visualizer.headset.n_blocks}")
    for i in range(visualizer.headset.n_blocks):
        block = visualizer.headset.get_block(i)
        print(f"Block {i}: {type(block)}")
        print(f'\tblock: {block}')